In [ ]:
%pip install geopy

Note: you may need to restart the kernel to use updated packages.



In [1]:
import geopy
from geopy.geocoders import Nominatim
import requests
import json
import random
import osmnx as ox
import geopandas as gpd
import math
overpass_url = "http://overpass-api.de/api/interpreter"
overpass_query = """
[out:json];
area["nat_ref"="63"]["admin_level"=4];
(node["amenity"="restaurant"](area);
 way["amenity"="restaurant"](area);
 rel["amenity"="restaurant"](area);
);
out center;
"""
response = requests.get(overpass_url,
                        params={'data':overpass_query})
data = response.json()

In [2]:
with open("restaurant-tpe.geojson", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
# Haversine 公式：計算兩個經緯度點的距離（單位：公里）
def haversine(lon1, lat1, lon2, lat2):
    R = 6371  # 地球半徑 (km)
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

# 載入 GeoJSON 檔案（假設餐廳資料存在這個檔案中）
with open("restaurant-tpe.geojson", encoding="utf-8") as f:
    data = json.load(f)

# 使用者輸入位置（例如：台北車站附近）
target_lat = 25.0478
target_lon = 121.5319

# 找出 1 公里內的所有餐廳
nearby_restaurants = []
for feature in data["features"]:
    lon, lat = feature["geometry"]["coordinates"]
    distance = haversine(target_lon, target_lat, lon, lat)
    if distance <= 1.0:
        nearby_restaurants.append(feature)

# 隨機選一間
if nearby_restaurants:
    selected = random.choice(nearby_restaurants)
    name = selected["properties"].get("name", "未命名餐廳")
    print(f"建議餐廳：{name}")
else:
    print("1 公里內沒有餐廳")

建議餐廳：中央市場生猛海鮮


In [4]:
geolocator = Nominatim(user_agent="geo_test")
location = geolocator.geocode("台大")
print(location.latitude, location.longitude)

25.0168175 121.5384571


In [3]:

# 設定地點名稱（可輸入中文）
place_name = "國立臺灣大學, 台北市, 台灣"

# 下載該區域的建築物資料（建築物是 landuse='university' 或 building=*）
tags = {'building': True}
buildings = ox.geometries_from_place(place_name, tags)

# 顯示建築物名稱與座標
for idx, row in buildings.iterrows():
    name = row.get('name', '未命名建築')
    if row.geometry.geom_type == 'Polygon':
        lon, lat = row.geometry.centroid.x, row.geometry.centroid.y
        print(f"{name}: 緯度 {lat}, 經度 {lon}")


c:\Users\asus\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-3-6d44873b9041>:6: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  buildings = ox.geometries_from_place(place_name, tags)


湖心亭: 緯度 25.020420324809436, 經度 121.53761438843765
綜合體育館: 緯度 25.021619723009035, 經度 121.53518090630956
nan: 緯度 25.01797184560771, 經度 121.53745009662846
nan: 緯度 25.018170400372455, 經度 121.53743918036332
小福: 緯度 25.018585121530528, 經度 121.53732401571276
農業陳列館: 緯度 25.018142098660174, 經度 121.53419815401914
新月台: 緯度 25.018549173230152, 經度 121.53401428538653
管理學院二號館: 緯度 25.013121459826277, 經度 121.53687679142193
資訊工程館: 緯度 25.019529742275587, 經度 121.54161874191405
社會科學院大樓: 緯度 25.020759003806415, 經度 121.54230762977845
雅頌坊: 緯度 25.013771496427733, 經度 121.53874565280383
管理學院交誼廳: 緯度 25.0134188839685, 經度 121.53838291787491
nan: 緯度 25.015197783452514, 經度 121.54147159879236
考種館: 緯度 25.016256149999997, 經度 121.54171149999999
二號館: 緯度 25.01677797792171, 經度 121.53591080867147
四號館: 緯度 25.016861783429857, 經度 121.53764669710031
土木系館: 緯度 25.01781210730518, 經度 121.53834234294453
三號館: 緯度 25.01649593394744, 經度 121.53590968199977
共同教學館: 緯度 25.01582701642889, 經度 121.53754666822215
思亮館國際會議廳: 緯度 25.02065128572151, 經度 12

第一版:輸入台大建築物，輸出方圓500公尺內隨機一間餐廳

In [2]:
import osmnx as ox

# 下載台大建築物資料
place_name = "國立臺灣大學, 台北市, 台灣"
buildings = ox.geometries_from_place(place_name, tags={'building': True})

# 儲存成 GeoJSON 格式
buildings.to_file("ntu_buildings.geojson", driver="GeoJSON")
print("建築物資料已儲存為 GeoJSON 格式。")


<ipython-input-2-8df7feacde5c>:5: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  buildings = ox.geometries_from_place(place_name, tags={'building': True})


建築物資料已儲存為 GeoJSON 格式。


In [4]:
import osmnx as ox
import geopandas as gpd
import requests
import random
from shapely.geometry import Point

c:\Users\asus\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [5]:
# 步驟 1: 下載台大建築物資料並儲存為 geojson 檔案
def download_buildings():
    place_name = "國立臺灣大學, 台北市, 台灣"
    buildings = ox.geometries_from_place(place_name, tags={'building': True})

    # 儲存為 GeoJSON 格式
    buildings.to_file("ntu_buildings.geojson", driver="GeoJSON")
    print("建築物資料已儲存為 GeoJSON 格式。")

In [6]:
import geopy
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import json
import random
import osmnx as ox
import geopandas as gpd
import math
overpass_url = "http://overpass-api.de/api/interpreter"
overpass_query = """
[out:json];
area["nat_ref"="63"]["admin_level"=4];
(node["amenity"="restaurant"](area);
 way["amenity"="restaurant"](area);
 rel["amenity"="restaurant"](area);
);
out center;
"""
response = requests.get(overpass_url,
                        params={'data':overpass_query})
data = response.json()

with open("restaurant-tpe.geojson", encoding="utf-8") as f:
    data = json.load(f)

In [7]:
# 計算兩點之間的距離（使用 Geopy 函式庫）
def haversine(lon1, lat1, lon2, lat2):
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers

In [8]:
# 載入餐廳的 GeoJSON 檔案
def load_restaurant_data(filename="restaurant-tpe.geojson"):
    with open(filename, encoding="utf-8") as f:
        data = json.load(f)
    return data

In [ ]:
# 搜尋 500 公尺內的餐廳
def find_nearby_restaurants(target_lat, target_lon, data, radius_km):
    nearby_restaurants = []
    for feature in data["features"]:
        lon, lat = feature["geometry"]["coordinates"]
        distance = haversine(target_lon, target_lat, lon, lat)
        if distance <= radius_km:
            nearby_restaurants.append(feature)
    return nearby_restaurants

In [11]:
# 隨機選擇餐廳
def recommend_random_restaurant(nearby_restaurants):
    if nearby_restaurants:
        selected = random.choice(nearby_restaurants)
        name = selected["properties"].get("name", "未命名餐廳")
        return f"建議餐廳：{name}"
    else:
        return "500 公尺內沒有餐廳"

In [12]:
# 主程式
def main():
    
    buildings = gpd.read_file("ntu_buildings.geojson")

    # 使用者輸入建築物名稱
    target_name = input("請輸入台大內的建築物名稱（例如：綜合教學館）：")

    # 模糊比對名稱找建築物
    matched = buildings[buildings['name'].str.contains(target_name, na=False)]
    if matched.empty:
        print("找不到這棟建築物")
        exit()

    # 取中心點作為目標經緯度
    target_geom = matched.iloc[0].geometry
    if target_geom.geom_type == "Polygon":
        target_point = target_geom.centroid
    else:
        target_point = target_geom

    target_lat = target_point.y
    target_lon = target_point.x
    # 載入餐廳資料
    restaurant_data = load_restaurant_data("restaurant-tpe.geojson")
    
    # 查詢附近餐廳
    nearby_restaurants = find_nearby_restaurants(target_lat, target_lon, restaurant_data,0.5)
    
    # 隨機選擇一間餐廳並顯示
    result = recommend_random_restaurant(nearby_restaurants)
    print(result)

In [15]:
if __name__ == "__main__":
    main()

Skipping field ways: unsupported OGR type: 1


建議餐廳：我家廚房早餐店


第二版:考慮時間、輸出一星期推薦餐廳表格

In [17]:
import geopy
from geopy.distance import geodesic
import json
import random
import geopandas as gpd
import datetime
import re

In [18]:
def haversine(lon1, lat1, lon2, lat2):
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers

In [19]:
def load_restaurant_data(filename="restaurant-tpe.geojson"):
    with open(filename, encoding="utf-8") as f:
        return json.load(f)

In [21]:

def is_open(restaurant, weekday, hour):
    hours = restaurant["properties"].get("opening_hours")
    if not hours:
        return False  # 沒有提供視為沒有營業

    # 將 weekday 轉成縮寫
    weekday_map = ["Mo", "Tu", "We", "Th", "Fr", "Sa", "Su"]
    day = weekday_map[weekday]

    # 解析 opening_hours，這裡做簡單的字串比對
    pattern = re.compile(rf"{day}.*?(\d{{2}}):(\d{{2}})-(\d{{2}}):(\d{{2}})")
    matches = pattern.findall(hours)
    for match in matches:
        start_h, start_m, end_h, end_m = map(int, match)
        start = start_h + start_m / 60
        end = end_h + end_m / 60
        if start <= hour <= end:
            return True
    return False

In [22]:
def find_nearby_restaurants(lat, lon, data, radius_km=0.5, weekday=0, hour=12):
    nearby = []
    for feature in data["features"]:
        lon2, lat2 = feature["geometry"]["coordinates"]
        if haversine(lon, lat, lon2, lat2) <= radius_km and is_open(feature, weekday, hour):
            nearby.append(feature)
    return nearby

In [23]:
def recommend_random_restaurant(nearby):
    if nearby:
        r = random.choice(nearby)
        return r["properties"].get("name", "未命名餐廳")
    return "無餐廳"

In [ ]:
def main():
    # 載入資料
    buildings = gpd.read_file("ntu_buildings.geojson")
    restaurant_data = load_restaurant_data()

    # 收集 10 個建築名稱
    input_names = []
    print("請輸入星期一到五午餐與晚餐建築名稱（共 10 筆）：")
    for i in range(10):
        name = input(f"第 {i+1} 筆：")
        input_names.append(name)

    # 每天午晚餐推薦表
    schedule = {"星期一": [], "星期二": [], "星期三": [], "星期四": [], "星期五": []}
    weekday_names = list(schedule.keys())

    for i, name in enumerate(input_names):
        matched = buildings[buildings['name'].str.contains(name, na=False)]
        if matched.empty:
            schedule[weekday_names[i // 2]].append("找不到建築物")
            continue

        geom = matched.iloc[0].geometry
        point = geom.centroid if geom.geom_type == "Polygon" else geom
        lat, lon = point.y, point.x
        hour = 12 if i % 2 == 0 else 18  # 午餐 or 晚餐
        nearby = find_nearby_restaurants(lat, lon, restaurant_data, weekday=i // 2, hour=hour)
        result = recommend_random_restaurant(nearby)
        schedule[weekday_names[i // 2]].append(result)

    # 輸出表格
    print("\n本週午晚餐推薦：")
    print("{:<10} {:<20} {:<20}".format("星期", "午餐", "晚餐"))
    for day in weekday_names:
        lunch, dinner = schedule[day]
        print("{:<10} {:>20} {:>20}".format(day, lunch, dinner))

In [29]:
if __name__ == "__main__":
    main() 

Skipping field ways: unsupported OGR type: 1


請輸入星期一到五午餐與晚餐建築名稱（共 10 筆）：

本週午晚餐推薦：
星期         午餐                   晚餐                  
Monday     站食可以                 石記牛肉麵               
Tuesday    嵐迪義大利麵館              靜壽司                 
Wednesday  石記牛肉麵                石記牛肉麵               
Thursday   壹森泰食堂                找到咖啡                
Friday     找到咖啡                 bb.q Chicken        
